This notebook allows the user to change defacing parameters to see impacts on the performance. Other than dependencies in `requirements.txt`, also install ipykernel, ipympl and ipywidgets.

**Interactive Mode**\
Create a config file with information regarding your input data, as instructed in the README.md file. Then, run the following cells in order. The final output will allow you to interact with different parameters to see its impacts on the output.

In [1]:
from interactive_backend import backend, backend_recompute
INPUT = 'config2.json'
nc, data, dataID, readout_axis, og_image_rsos = backend(INPUT)

[UPDATE] Data has been successfully loaded
[UPDATE] Nifti file successfully saved to input/input_image_t1_01.nii.gz
Mask saved successfully


In [2]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
from ipywidgets import interact, IntSlider, Dropdown, Dropdown, Output, VBox, Button
import os
from scipy.linalg import orth
from scipy.linalg import norm
from ROVir import form_virtual_coil_data, rovir
from raw_deface_opt import set_masks, make_A_B
import numpy as np
from IPython.display import clear_output, display
import nibabel as nib

# make a save button for display image
save_button = Button(description='Save current view',button_style='success')

# make slider for selecting top virtual coils
n_coils_slider = IntSlider(value=1, min=1, 
                    max=nc, step=1, 
                    description='Number of Top Virtual Coils', 
                    continuous_update=False, 
                    style={'description_width': 'initial'},
                    layout={'width': '600px'})

# make sliders for x, y, z slices of the brain
x_slicer = IntSlider(value=data.shape[0]//2, min=0,
                     max = data.shape[0]-1, step=1,
                     description='Sagittal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

y_slicer = IntSlider(value=data.shape[1]//2, min=0,
                     max = data.shape[1]-1, step=1,
                     description='Coronal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

z_slicer = IntSlider(value=data.shape[2]//2, min=0,
                     max = data.shape[2]-1, step=1,
                     description='Axial Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

# make drop down box to select masking options
mask_selector = Dropdown(options=['Default', 'A', 'B', 'C', 'D', 'E', 'F'],
                         value='Default', description='Masking Scheme',
                         style={'description_width': 'initial'},
                         layout={'width': '300px'})

# make slider to choose gap between brain and face masks
gap_slider = IntSlider(value=10, min=0, max=30, 
                       step=1, description='Brain and Face Mask Gap',
                       continuous_update=False, 
                       style={'description_width': 'initial'},
                       layout={'width': '600px'})



In [7]:
plot_output = Output()
save_status = Output()
cache = {} # use dictionary to cache already computed images
last_render = {}
max_cache = 10

def recompute_output(n_coils, mask_option, gap):

    cache_key = f'{dataID}_{n_coils}_{mask_option}_{gap}'
    if cache_key in cache:
        return cache[cache_key]

    maskA, maskB, image_rsos, brain_retain, face_retain = backend_recompute(nc, data, dataID, readout_axis, mask_option, gap, n_coils)
    
    ratio = image_rsos/og_image_rsos # take the ratio between the defaced and original image

    result = {
        'maskA': maskA,
        'maskB': maskB,
        'image_rsos': image_rsos,
        'ratio': ratio,
        'brain_retain': brain_retain,
        'face_retain': face_retain    
    }

    cache[cache_key] = result

    # clear least recent cache
    if len(cache) > max_cache:
        oldest_key = next(iter(cache))
        removed = cache.pop(oldest_key)
    
    return result

def reslice():
    n_coils = n_coils_slider.value
    mask_option = mask_selector.value
    gap = gap_slider.value 
    x, y, z = x_slicer.value, y_slicer.value, z_slicer.value

    with plot_output:
        clear_output(wait=True)
        display('RECOMPUTING DEFACED IMAGES...')

    with save_status:
        clear_output(wait=True)

    save_button.disabled = True
    
    # recompute output or get from cache
    result = recompute_output(n_coils, mask_option, gap)

    # unpack the results
    maskA = result['maskA']
    maskB = result['maskB']
    image_rsos = result['image_rsos']
    ratio = result['ratio']
    brain_retain = result['brain_retain']
    face_retain = result['face_retain']

    with plot_output:
        clear_output(wait=True)
        plt.close('all')

        # plot original, mask overlayed, and defaced images
        fig = plt.figure(figsize=(15, 10))
    
        plt.subplot(4,3,1)
        plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')
    
        plt.subplot(4,3,2)
        plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')
    
        plt.subplot(4,3,3)
        plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')
    
        plt.subplot(4,3,4)
        plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')
        plt.imshow(np.rot90(maskB[:, :, z]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[:, :, z]), alpha = 0.3, cmap = 'Greens')
        
        plt.subplot(4,3,5)
        plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')
        plt.imshow(np.rot90(maskB[x, :, :]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[x, :, :]), alpha = 0.3, cmap = 'Greens')
        
        plt.subplot(4,3,6)
        plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')
        plt.imshow(np.rot90(maskB[:, y, :]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[:, y, :]), alpha = 0.3, cmap = 'Greens')
    
        plt.subplot(4,3,7)
        plt.imshow(np.rot90(image_rsos[:, :, z]), cmap='gray')
    
        plt.subplot(4,3,8)
        plt.imshow(np.rot90(image_rsos[x, :, :]), cmap='gray')
    
        plt.subplot(4,3,9)
        plt.imshow(np.rot90(image_rsos[:, y, :]), cmap='gray')
        
        global_vmin = np.min([ratio[:, :, z].min(), ratio[x, :, :].min(), ratio[:, y, :].min()])
        global_vmax = np.max([ratio[:, :, z].max(), ratio[x, :, :].max(), ratio[:, y, :].max()])
    
        plt.subplot(4,3,10)
        ratio_img1 = plt.imshow(np.rot90(ratio[:, :, z]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        plt.subplot(4,3,11)
        ratio_img2 = plt.imshow(np.rot90(ratio[x, :, :]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        plt.subplot(4,3,12)
        ratio_img3 = plt.imshow(np.rot90(ratio[:, y, :]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        ratio_colourbar_ax = plt.axes([0.25, 0.05, 0.5, 0.015]) 
        ratio_colourbar = plt.colorbar(ratio_img3, cax=ratio_colourbar_ax, orientation='horizontal')
        ratio_colourbar.set_label('Retention Ratio', fontsize=10)
    
        plt.show()
    
        print(f'Brain region signal retention: {brain_retain}')
        print(f'Face region signal retention: {face_retain}')

    last_render['fig'] = fig
    last_render['params'] = (n_coils, mask_option, gap, x, y, z)
    save_button.disabled = False
    plt.close(fig)

In [5]:
def on_slider_change(change):
    reslice()

def on_save_clicked(b):
    if 'fig' not in last_render:
        return

    os.makedirs('results', exist_ok=True)
    n_coils, mask_option, gap, x, y, z = last_render['params']
    filename = f"results/{dataID}_ncoils{n_coils}_mask{mask_option}_gap{gap}_x{x}_y{y}_z{z}.png"
    last_render['fig'].savefig(filename, dpi=150, bbox_inches='tight')

    with save_status:
        clear_output(wait=True)
        print(f'Saved to {filename}')

save_button.on_click(on_save_clicked)
    
for widget in (n_coils_slider, mask_selector, gap_slider, x_slicer, y_slicer, z_slicer):
    widget.observe(on_slider_change, names='value')

reslice() # initial images generation 

# display widgets onto the screen
display(VBox([n_coils_slider, mask_selector, gap_slider, x_slicer, y_slicer, z_slicer, plot_output, save_button, save_status]))